Hiperparâmetros a explorar e justificar no relatório: a equipe deve treinar pelo menos
duas configurações distintas (variando, por exemplo, rank 4 vs. 8, learning rate, ou número
de passos) e comparar os resultados. Decorar valores não basta — o barema avalia a
justificativa da escolha final.

    • rank: dimensão das matrizes LoRA — valores maiores capturam mais detalhes do
    estilo, mas aumentam o risco de sobreajuste com datasets pequenos;

    • max_train_steps: poucos passos = estilo fraco; passos demais = modelo
    “decora” as imagens (overfitting). Inspecionem as imagens de validação a cada
    checkpoint;

    • learning_rate: taxas altas desestabilizam o treino; típico para LoRA: 1e-4 a 5e-5;

    • fp16: precisão reduzida indispensável para caber na VRAM de 16 GB da T4.

In [ ]:
!pip -q install diffusers transformers accelerate peft datasets

!git clone https://github.com/huggingface/diffusers

%cd diffusers/examples/text_to_image

from huggingface_hub import login

login() # cole o token (nunca o escreva no código!)


In [ ]:
!accelerate launch train_text_to_image_lora.py \
--pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5"
\
 --train_data_dir="/content/dataset" \
 --resolution=512 --train_batch_size=1 \
 --gradient_accumulation_steps=4 \
 --max_train_steps=1500 \
 --learning_rate=1e-4 --lr_scheduler="cosine" \
 --rank=8 \
 --mixed_precision="fp16" \
 --checkpointing_steps=500 \
 --validation_prompt="estilo_cordel, um vaqueiro tocando viola" \
 --output_dir="/content/lora_saida" \
 --push_to_hub --hub_model_id="equipe/lora-estilo-cordel"